# DeepMeow — Computer Vision Research Notebook (Google Colab)

Welcome to the experiment workspace for **DeepMeow**, a research project focused on building a single-shot object detector and multi-object tracker from first principles in PyTorch.

### Pipeline Overview:
1. **Hardware Verification**: Verify CUDA GPU availability (Tesla T4 or better).
2. **Repository Synchronization**: Pull the latest modular codebase directly from GitHub.
3. **Persistent Storage**: Mount Google Drive to cache raw images (~500 MB) across runtime sessions.
4. **Data Acquisition**: Execute `downloader.py` to filter and extract cat instances from COCO 2017.
5. **Dataset Validation**: Verify image-annotation integrity and bounding box counts.
6. **Visual Inspection**: Overlay ground-truth bounding boxes (`[x, y, w, h]` -> `[x1, y1, x2, y2]`) on sample images.
7. **Backbone Verification**: Pass a dummy batch through our custom ResNet-style backbone to verify output tensor dimensions ($P_3, P_4, P_5$).

> **Note for Collaborators**: Ensure your Colab runtime accelerator is set to GPU (`Runtime -> Change runtime type -> T4 GPU`).

## 1. Hardware Initialization & GPU Verification

In [9]:
import torch

# Check CUDA device availability for PyTorch operations
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU available: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    print('WARNING: No GPU detected. Please switch runtime to GPU (Runtime -> Change runtime type -> T4 GPU).')

GPU available: Tesla T4 (15.6 GB VRAM)


## 2. Environment Setup & Repository Synchronization

In [10]:
import os

REPO_URL = 'https://github.com/IliyaJz/DeepMeow.git'
REPO_DIR = '/content/DeepMeow'

# Synchronize remote codebase with local Colab runtime environment
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Pulling latest changes...')
    !git -C {REPO_DIR} pull

# Change directory to repository root for relative module imports
os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')

Repository already cloned. Pulling latest changes...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 11 (delta 7), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 4.76 KiB | 1.19 MiB/s, done.
From https://github.com/IliyaJz/DeepMeow
   97d5098..ec33803  main       -> origin/main
Updating 97d5098..ec33803
Fast-forward
 notebooks/DeepMeow_Colab.ipynb | 107 +++++++++++++++++++++++++++++++++--------
 src/data/downloader.py         |  15 ++++--
 2 files changed, 97 insertions(+), 25 deletions(-)

Working directory: /content/DeepMeow


In [11]:
# Install project dependencies (pycocotools, albumentations, opencv, pyyaml)
print('Installing dependencies...')
!pip install -q -r requirements.txt
print('All packages installed!')

Installing dependencies...
All packages installed!


## 3. Persistent Storage Setup (Google Drive)

Mounting Google Drive allows us to persist downloaded datasets and training checkpoints across Colab session disconnects.

In [12]:
USE_DRIVE = True  # Enable persistent storage on Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted!')
    print('Data will be saved to Google Drive.')
else:
    print('Skipping Drive mount. Data will be stored in Colab /content/ (resets each session).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted!
Data will be saved to Google Drive.


## 4. Dataset Acquisition & COCO Class Filtering

We execute `downloader.py` which:
1. Downloads the COCO 2017 instance annotation package (`annotations_trainval2017.zip`, ~241 MB).
2. Filters all annotations for `category_id == 17` (cat category) and re-maps the ID to `1`.
3. Downloads ~3,000 training images and ~500 validation images from the official COCO image servers.

In [13]:
# Execute streaming downloader script
!python src/data/downloader.py

Saving data to Google Drive: /content/drive/MyDrive/DeepMeow/data
 DeepMeow Dataset Downloader
 Target folder: /content/drive/MyDrive/DeepMeow/data

Downloading: annotations_trainval2017.zip
  Progress: 100% 253M/253M [00:05<00:00, 42.8MB/s]
  Saved to /content/drive/MyDrive/DeepMeow/data/tmp/annotations_trainval2017.zip

Extracting annotations ZIP...
  Extracted.

Filtering COCO 'train' annotations for cats...
  3000 images, 3444 annotations
  Saved -> /content/drive/MyDrive/DeepMeow/data/annotations/train.json

  train: 100% 3000/3000 [19:44<00:00,  2.53it/s]
  Images saved to: /content/drive/MyDrive/DeepMeow/data/raw/train

Filtering COCO 'val' annotations for cats...
  184 images, 202 annotations
  Saved -> /content/drive/MyDrive/DeepMeow/data/annotations/val.json

  val: 100% 184/184 [01:11<00:00,  2.57it/s]
  Images saved to: /content/drive/MyDrive/DeepMeow/data/raw/val

Cleaning up temporary files...
  Done!

 Dataset ready!
    Train images : /content/drive/MyDrive/DeepMeow/dat

## 5. Dataset Validation & Integrity Check

We inspect the generated JSON annotation files and verify that the downloaded `.jpg` image files match our annotation records.

In [14]:
import json
from pathlib import Path

# Determine dataset root path (Drive vs. local fallback)
drive_data = Path('/content/drive/MyDrive/DeepMeow/data')
local_data = Path('data')

data_root = drive_data if (USE_DRIVE and (drive_data / 'annotations/train.json').exists()) else local_data
print(f'Using dataset root: {data_root.resolve()}\n')

for split in ['train', 'val']:
    ann_path   = data_root / f'annotations/{split}.json'
    image_dir  = data_root / f'raw/{split}'

    if ann_path.exists():
        with open(ann_path) as f:
            ann = json.load(f)

        n_images      = len(ann['images'])
        n_annotations = len(ann['annotations'])
        n_files       = len(list(image_dir.glob('*.jpg')))

        print(f'{split.upper()}:')
        print(f'  Annotation images : {n_images}')
        print(f'  Annotation boxes  : {n_annotations}')
        print(f'  Downloaded files  : {n_files} .jpg files')
        print()
    else:
        print(f'{split.upper()}: Annotations file not found at {ann_path}')

Using dataset root: /content/drive/MyDrive/DeepMeow/data

TRAIN:
  Annotation images : 3000
  Annotation boxes  : 3444
  Downloaded files  : 3000 .jpg files

VAL:
  Annotation images : 184
  Annotation boxes  : 202
  Downloaded files  : 184 .jpg files



## 6. Ground-Truth Bounding Box Inspection

To verify our data pipeline before training, we render 6 randomly selected training images overlaid with their ground-truth bounding box coordinates (`[x_min, y_min, width, height]`).

In [15]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/DeepMeow/data')
local_data = Path('data')
data_root  = drive_data if (USE_DRIVE and (drive_data / 'annotations/train.json').exists()) else local_data

ann_file  = data_root / 'annotations/train.json'
train_dir = data_root / 'raw/train'

if ann_file.exists():
    with open(ann_file) as f:
        ann_data = json.load(f)

    id_to_anns = {}
    for ann in ann_data['annotations']:
        id_to_anns.setdefault(ann['image_id'], []).append(ann)

    id_to_img = {img['id']: img for img in ann_data['images']}

    # Select valid images with existing disk files
    valid_ids = [
        img_id for img_id in id_to_anns
        if (train_dir / id_to_img[img_id]['file_name']).exists()
    ]
    sample_ids = random.sample(valid_ids, min(6, len(valid_ids)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('COCO Cat Dataset — Ground-Truth Bounding Box Verification', fontsize=14)

    for ax, img_id in zip(axes.flat, sample_ids):
        img_info = id_to_img[img_id]
        img_path = train_dir / img_info['file_name']
        
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Image ID: {img_id} | Resolution: {img_info['width']}x{img_info['height']}", fontsize=9)
        
        # Draw bounding boxes
        for ann in id_to_anns.get(img_id, []):
            x, y, w, h = ann['bbox']
            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2, edgecolor='#FF6B35', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x, y - 4, 'cat', color='#FF6B35', fontsize=8, fontweight='bold')

    os.makedirs('results', exist_ok=True)
    plt.tight_layout()
    plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sample visualization saved to results/sample_images.png')

<Figure size 1500x1000 with 6 Axes>

## 7. Custom CNN Backbone Forward-Pass & Feature Map Verification

We test our custom `Backbone` module (`src/models/backbone.py`) with a dummy batch `[B=2, C=3, H=416, W=416]` to ensure proper channel dimensions and spatial receptive field reduction across multi-scale feature maps ($P_3, P_4, P_5$).

In [16]:
import torch
from src.models.backbone import Backbone

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Instantiate custom ResNet-style backbone architecture
backbone = Backbone().to(device)

# Compute total trainable parameters
total_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f'Backbone parameters: {total_params:,}')

# Forward pass verification with dummy input tensor [2, 3, 416, 416]
dummy_tensor = torch.randn(2, 3, 416, 416, device=device)

with torch.no_grad():
    p3, p4, p5 = backbone(dummy_tensor)

print(f'\nFeature map shapes:')
print(f'  P3 (small objects) : {tuple(p3.shape)}   <- 52x52 grid (stride 8)')
print(f'  P4 (medium objects): {tuple(p4.shape)}  <- 26x26 grid (stride 16)')
print(f'  P5 (large objects) : {tuple(p5.shape)}  <- 13x13 grid (stride 32)')
print('\nBackbone forward pass OK!')

Using device: cuda
Backbone parameters: 40,584,928

Feature map shapes:
  P3 (small objects) : (2, 256, 52, 52)   <- 52x52 grid (stride 8)
  P4 (medium objects): (2, 512, 26, 26)  <- 26x26 grid (stride 16)
  P5 (large objects) : (2, 1024, 13, 13)  <- 13x13 grid (stride 32)

Backbone forward pass OK!


## 8. Milestone Summary & Next Steps

### Week 1 Milestone Status:
- **Data Loader**: Verified dataset annotations and verified bounding box transformations.
- **Backbone Architecture**: Implemented custom ResNet-style feature extractor with multi-scale outputs ($P_3, P_4, P_5$).

### Week 2 Planned Modules:
1. **Feature Pyramid Network (FPN)**: Combine top-down semantics with bottom-up details across scales.
2. **Anchor Box Generator**: Generate multi-scale anchor reference boxes for $52\times52$, $26\times26$, and $13\times13$ feature maps.
3. **Detection Head & Multi-Task Loss**: Implement prediction heads and Complete IoU (CIoU) + Focal Loss functions.